# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Get overview of record sets and their fields
print("Record sets in the dataset (by @id):")
for record_set in metadata.record_sets:
    print(f"- RecordSet @id: {record_set['@id']} | name: {record_set['name']}")
    print("  Fields:")
    for field in record_set.get('fields', []):
        print(f"    - Field @id: {field['@id']}, name: {field.get('name', '')}, dataType: {field.get('dataType', '')}, source: {field.get('source', '')}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Example: extract all data from the first record set
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Columns in {record_set_id}:", dataframes[record_set_id].columns.tolist())
        display(dataframes[record_set_id].head())
    else:
        print(f"No records found for {record_set_id}.")

# For example, select the main record set if available
main_record_set = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a numeric field for analysis from the main record set

if main_record_set:
    df = dataframes[main_record_set]
    # List numeric columns (inferred by dtype)
    numeric_fields = df.select_dtypes(include='number').columns.tolist()
    print("Numeric fields detected:", numeric_fields)
    
    # Example: work with the first numeric field if available
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Examining field: {numeric_field}")
        # Filter records (e.g., values greater than a threshold)
        threshold = df[numeric_field].quantile(0.75) # Example: use 75th percentile as threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field (z-score)
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Optional: group by a categorical field if one exists
        cat_fields = df.select_dtypes(include=['object']).columns.tolist()
        group_field = cat_fields[0] if cat_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric field detected for EDA.")
else:
    print("No main record set found or loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set and numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} distributions grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped: no numeric field or record set available.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.